# Post-processing a remage simulation

This tutorial starts from the energy depositions of a _remage_ simulation and
works out what the detectors would have measured: the energy left in the active
volume, its resolution, the drift time of the charges and the pulse shape. It
writes the result to disk, groups the detectors into events, and uses the liquid
argon around them as a veto.

The simulation is close to the one of the
[remage tutorial](https://remage.readthedocs.io/en/stable/tutorial.html): a BEGe and an
inverted-coaxial detector in liquid argon, with a $^{228}$Th source between
them, and only the events leaving at least 1 MeV in germanium kept. Its output
and its geometry come from
[legend-testdata](https://github.com/legend-exp/legend-testdata).

The whole dataset is small enough to be held in memory, which is the simplest
way to work. The last section shows what changes when it is not.

In [ ]:
# the remage simulation and the detector maps, from legend-testdata
from legendtestdata import LegendTestData

ldata = LegendTestData()
ldata.checkout("0e9e3de")
testdata = ldata.get_path("remage")

stp_file = f"{testdata}/tutorial/th228-stp.lh5"
gdml_file = f"{testdata}/tutorial/geometry.gdml"
dtmap_file = f"{testdata}/V99999Z-3500V-hpge-drift-time-map.lh5"
psl_file = f"{testdata}/V99999Z-3500V-hpge-pulse-shape-lib.lh5"
hit_file = "th228_hit.lh5"

## The detector geometry

The processors that model the germanium detectors need to know their shape and
where they sit. Both come from the same GDML file that _remage_ simulated, read
back with [pyg4ometry](https://pyg4ometry.readthedocs.io). The detector
metadata stored in it by
[legend-pygeom-tools](https://legend-pygeom-tools.readthedocs.io) is turned
into an HPGe object by
[legend-pygeom-hpges](https://legend-pygeom-hpges.readthedocs.io).

_remage_ also writes the position of every detector to its output file, in the
`detector_origins` table, which is an alternative when the GDML file is not at
hand.

In [ ]:
import pyg4ometry
from pygeomhpges import make_hpge
from pygeomtools.detectors import get_sensvol_metadata

registry = pyg4ometry.gdml.Reader(gdml_file).getRegistry()

hpges = {
    name: make_hpge(get_sensvol_metadata(registry, name), registry=None)
    for name in ("BEGe", "ICPC")
}
positions = {name: registry.physicalVolumeDict[name].position for name in hpges}

{name: f"{hpge.mass:.0f} g" for name, hpge in hpges.items()}

## Reading the data

We read the detector table as an LGDO [Table](https://legend-pydataobj.readthedocs.io/en/stable/api/lgdo.types.html#lgdo.types.table.Table) and view
it as an [Awkward](https://awkward-array.org) array. The Awkward view is what
the processors work on: one table row per hit (i.e. the collection of steps in the detector), and inside it an array of steps. The
`with_units` flag attaches the physical units of each field, which the
processors read and convert as needed.

In [ ]:
import awkward as ak
import lh5

stp = lh5.read("stp/ICPC", stp_file)
data = stp.view_as("ak", with_units=True)

data.type.show()

Inspect the second detected event in the "ICPC" detector, which consists of three steps:

In [ ]:
data[1].show()

## Applying processors

A _reboost_ "processor" takes columns of the _remage_ output and computes one new quantity per detector hit.

### The inactive layer

The first one we apply is the correction for the inactive layer at the surface
of a germanium detector: a charge deposited close to the n+ electrode is only
partly collected, so the measured energy is lower than the deposited one.

`distance_to_surface` gives the distance of every step to the surface, and
`piecewise_linear_activeness` turns it into the fraction of charge that is
collected there. Summing the weighted energies over the steps of a hit gives
the energy the detector would report.

In [ ]:
from reboost.hpge import surface
from reboost.math import functions

distance = surface.distance_to_surface(
    data.xloc, data.yloc, data.zloc, hpges["ICPC"], positions["ICPC"]
)

# 1.5 mm of inactive layer, the outermost 0.3 mm of which is fully dead
activeness = functions.piecewise_linear_activeness(distance, fccd_in_mm=1.5, dlf=0.2)

deposited = ak.sum(data.edep, axis=-1)
collected = ak.sum(data.edep * activeness, axis=-1)

In [ ]:
import hist
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4))
for values, label in ((deposited, "deposited"), (collected, "collected")):
    h = hist.new.Reg(160, 1000, 3400, name="energy [keV]").Double().fill(values)
    h.plot(ax=ax, yerr=False, flow="none", label=label)
ax.set_yscale("log")
ax.set_xlim(1000, 3400)
ax.set_ylabel("counts / 20 keV")
_ = ax.legend()

The correction moves counts out of the full-energy peaks into the continuum
below them. Most hits end up far below the range plotted here.

### Energy resolution

The simulation assumes infinite energy resolution, so the peaks are single bins wide.
`gaussian_sample` smears each energy with the detector resolution.

In [ ]:
from reboost.math import stats

energy = stats.gaussian_sample(collected, sigma=1.5, seed=1234)  # unit is keV
hist.new.Reg(100, 2605, 2625, name="energy [keV]").Double().fill(energy)

### Drift time

The drift time is how long the holes take to reach the p$^+$ contact. We read it
from a map of the $(r, z)$ plane of the detector, computed with [SolidStateDetectors.jl](https://juliaphysics.github.io/SolidStateDetectors.jl/stable/)
at the operating voltage of 3500 V, with the p$^+$ contact at the origin. Holes
drift faster along some crystal axes than along others, so there is one map for
the $\langle 100 \rangle$ axis and one for the $\langle 110 \rangle$ axis. The file is sourced from the LEGEND test data.

In [ ]:
from reboost.hpge import load_hpge_drift_time_maps, plot_drift_time_maps

dt_maps = load_hpge_drift_time_maps(dtmap_file, "V99999Z")
fig, _ = plot_drift_time_maps(dt_maps, hpge=hpges["ICPC"], title="V99999Z", ratio_vmax=1.2)

The drift time grows with the distance from the p+ contact, up to about 2 µs at
the top corners. The two axes differ by a few percent, which the ratio panel
shows.
`drift_time_crystal_axes` interpolates between the two maps
at the position of every step, taking the $\langle 100 \rangle$ axis along $x$.
A hit gets the mean of its steps, weighted by their energy.

In [ ]:
from reboost.hpge import psd

step_drift_time = psd.drift_time_crystal_axes(
    data.xloc, data.yloc, data.zloc, dt_maps, coord_offset=positions["ICPC"]
)
drift_time = ak.sum(step_drift_time * data.edep, axis=-1) / ak.sum(data.edep, axis=-1)

hist.new.Reg(100, 0, 2500, name="drift time [ns]").Double().fill(drift_time)

Most hits sit around 1 µs, on the side of the detector facing the source, 20 to
30 mm above the p$^+$ contact. The tail up to 2 µs comes from the upper part of the
detector. A hit spread over several sites mixes drift times, which the weighted
mean hides.

### A/E from a single template

A $\gamma$ particle absorbed in one spot makes a short, high current pulse; one that
scatters several times makes a lower and wider one. Experiments separate the two
with A/E, the maximum of the current divided by the energy.

The _remage_ simulation has no waveforms, so `maximum_current` sums
one template current pulse per step, scaled by the energy of the step and placed at its
drift time, and takes the maximum. The template comes from
`get_current_template`, an analytic current pulse with
tunable parameters. Normalised to a maximum of one, it puts $A/E$ at one for
a hit deposited at a single drift time, and below one for a hit spread over
several sites.

In [ ]:
from reboost.hpge import get_current_template, maximum_current

template, times = get_current_template(
    low=-1000,
    high=4000,
    step=1,
    mean_aoe=1,  # maximum of the template, so that A/E is normalised
    amax=1250.0,
    mu=5.0,
    sigma=45.0,
    tail_fraction=0.35,
    tau=150.0,
    high_tail_fraction=0.10,
    high_tau=80.0,
)

a_max = maximum_current(data.edep * activeness, step_drift_time, template=template, times=times)
aoe = a_max / energy

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for lo, hi, label in (
    (1587.5, 1597.5, "1592 keV, double escape"),
    (2609.5, 2619.5, "2615 keV, full energy"),
):
    h = hist.new.Reg(240, 0.3, 1.26, name="A/E").Double().fill(aoe[(energy > lo) & (energy < hi)])
    h.plot(ax=ax, yerr=False, flow="none", density=True, label=label)
ax.set_ylabel("normalised counts")
ax.set_yscale("log")
_ = ax.legend()

The $^{228}$Th double-escape peak at 1592 keV is single-site and piles up at one. The
2615 keV full-energy peak, where the gamma scatters before being absorbed,
spreads to lower values: a cut on A/E keeps the first and rejects most of the
second.

### A/E from a pulse-shape library

The template above is the same everywhere in the detector. A pulse-shape library
instead holds one pulse per position, from the same field simulation that
produced the drift-time map. In this tutorial we use a pulse-shape library from the LEGEND test data. It stores charge pulses, so we take their derivative to get the current.

In [ ]:
import numpy as np

from reboost.hpge import HPGePulseShapeLibrary, plot_psl_aoe_maps

lib = lh5.read("V99999Z", psl_file)
dt = lib["dt"].value

libraries = {}
for angle in (0, 45):
    charge = lib[f"waveform_{angle:03d}_deg"].view_as("np")
    libraries[angle] = HPGePulseShapeLibrary(
        np.diff(charge, axis=-1) / dt,
        lib["r"].attrs["units"],
        lib["z"].attrs["units"],
        lib["dt"].attrs["units"],
        lib["r"].view_as("np"),
        lib["z"].view_as("np"),
        np.arange(charge.shape[-1] - 1) * dt,
    )

fig, _ = plot_psl_aoe_maps(libraries, hpge=hpges["ICPC"], title="V99999Z", vmin=0.9, vmax=3)

The $A/E$ of every grid point is the maximum of its current pulse.
`plot_psl_aoe_maps` maps it for both crystal axes, normalised to
the most common value, the one of the flat bulk. $A/E$ rises steeply only for the
charges created next to the p$^+$ contact, which give the high-$A/E$ events of a real
experiment.

To use the library, `maximum_current` needs each pulse placed at the drift time
of its step, so we shift every pulse to put its maximum at the origin of the
time axis. It reads the pulses at whatever sampling they were stored with, 8 ns
here. The step coordinates are given in the frame of the detector, as for
the drift-time map.

In [ ]:
# the A/E of the bulk is the most common one, normalising by it puts the bulk at
# one, as in the map above
current = np.nan_to_num(libraries[0].waveforms)
peaks = current.max(axis=-1)
counts, edges = np.histogram(peaks[peaks > 0], bins=1000)
current /= ((edges[:-1] + edges[1:]) / 2)[counts.argmax()]

# every pulse is shifted to put its maximum at the origin of the time axis

n = current.shape[-1]
peak_at = int(1000 / dt)  # keep 1 µs of pulse before the maximum
peak = np.argmax(current, axis=-1)
idx = peak[..., None] + np.arange(n) - peak_at
shifted = np.where(
    (idx >= 0) & (idx < n), np.take_along_axis(current, np.clip(idx, 0, n - 1), axis=-1), 0.0
)
library = libraries[0]._replace(
    waveforms=shifted.astype(np.float32), t=(np.arange(n) - peak_at) * dt
)

x0, y0, z0 = positions["ICPC"].eval()
r_step = np.sqrt((data.xloc - x0 / 1000) ** 2 + (data.yloc - y0 / 1000) ** 2)
z_step = data.zloc - z0 / 1000

aoe_psl = (
    maximum_current(
        data.edep * activeness, step_drift_time, r=r_step, z=z_step, template=library, times=None
    )
    / energy
)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for lo, hi, label in (
    (1587.5, 1597.5, "1592 keV, double escape"),
    (2609.5, 2619.5, "2615 keV, full energy"),
):
    sel = (energy > lo) & (energy < hi)
    h = hist.new.Reg(240, 0.3, 1.26, name="A/E").Double().fill(aoe_psl[sel])
    h.plot(ax=ax, yerr=False, flow="none", density=True, label=label)
ax.set_ylabel("normalised counts")
ax.set_yscale("log")
_ = ax.legend()

The two peaks separate as with the template, and the single-site peak sits at
one because of the normalisation. The library adds what a single template cannot
know: the pulse shape, and with it A/E, depends on where the charge was
created.

A few hits now sit above one. They come from close to the p+
contact, where the map above rises.

## Writing the hit file

`init_hit_table` starts an output table from the input one. It carries over the
two fields that identify a hit in time, the Geant4 event identifier `evtid` and
the time of the hit `t0`, and nothing else. We add the quantities we computed
and write the table with `write_hit_table_chunk`.

The `uid` is the number the detector was registered with in the _remage_ macro
(`th228.mac`, next to the input file). Passing it makes the function link the
table under `hit/__by_uid__/det002`, the same layout _remage_ uses, so that the
tools that look detectors up by identifier also work on our file.

In [ ]:
from pathlib import Path

from lgdo import Array

from reboost import io

hit = io.init_hit_table(stp)
hit.add_field("energy", Array(energy, attrs={"units": "keV"}))
hit.add_field("drift_time", Array(drift_time, attrs={"units": "ns"}))
hit.add_field("aoe", Array(aoe))

# the function appends, so remove the output of an earlier run
Path(hit_file).unlink(missing_ok=True)

io.write_hit_table_chunk(hit, "hit/ICPC", hit_file, uid=2)

The other two detectors go the same way: the same processors for the BEGe, minus
the drift time and A/E, which need a map, and for the argon only the summed
energy. `get_remage_detector_uids` lists the detectors in
the input file.

In [ ]:
from reboost import utils

uids = utils.get_remage_detector_uids(stp_file)

# start from a clean file, the writer appends to what it finds
Path(hit_file).unlink(missing_ok=True)

for uid, name in uids.items():
    stp = lh5.read(f"stp/{name}", stp_file)
    data = stp.view_as("ak", with_units=True)
    out = io.init_hit_table(stp)

    if name in hpges:
        distance = surface.distance_to_surface(
            data.xloc, data.yloc, data.zloc, hpges[name], positions[name]
        )
        activeness = functions.piecewise_linear_activeness(distance, fccd_in_mm=1.5, dlf=0.2)
        collected = ak.sum(data.edep * activeness, axis=-1)
        out.add_field(
            "energy",
            Array(
                stats.gaussian_sample(collected, sigma=1.5, seed=1234),
                attrs={"units": "keV"},
            ),
        )
        if name == "ICPC":
            step_drift_time = psd.drift_time_crystal_axes(
                data.xloc, data.yloc, data.zloc, dt_maps, coord_offset=positions[name]
            )
            drift_time = ak.sum(step_drift_time * data.edep, axis=-1) / ak.sum(data.edep, axis=-1)
            out.add_field("drift_time", Array(drift_time, attrs={"units": "ns"}))
            a_max = maximum_current(
                data.edep * activeness, step_drift_time, template=template, times=times
            )
            out.add_field("aoe", Array(a_max / out["energy"].view_as("np")))
    else:
        out.add_field("energy", Array(ak.sum(data.edep, axis=-1), attrs={"units": "keV"}))

    io.write_hit_table_chunk(out, f"hit/{name}", hit_file, uid=uid)

lh5.show(hit_file)

The germanium tables and the argon table have different fields, which is fine:
each detector table stands on its own.

## Event analysis

Everything so far worked on one detector at a time. A processor sees the hits of
a single table and returns a new column for it, and nothing in it depends on
what the other detectors were doing. This is where the detector models live: the
surface response, the energy resolution, the pulse shape.

An experiment, though, does not record detectors, it records events: what all
the detectors saw at the same time. Going from the one to the other is the
second half of the post-processing, and it starts from the time-coincidence map
(TCM). Each row of the TCM is an event, and lists the hits, across all
detectors, that happened in the same Geant4 event and close enough in time to be
read out together.

_remage_ writes one next to the steps, and it describes our file just as well:
the processors did not change the number of rows nor their order, so row $i$ of
`hit/ICPC` is the same hit as row $i$ of `stp/ICPC`.

In [ ]:
tcm = lh5.read_as("tcm", stp_file, "ak")
tcm[:5].show(limit_cols=200)

> **When to rebuild it.** The map points at rows, so it only holds as long as
> those rows do. Rebuild it with `reboost.tcm.build_remage_tcm` if you drop or
> reorder hits, if you put several files together, or if you want a coincidence
> window other than the one _remage_ used. It writes the map into the file you
> give it, next to the detector tables:
>
> ```python
> reboost.tcm.build_remage_tcm(hit_file, hit_file, coin_window_in_ns=1000)
> ```

`table_key` is the detector identifier and `row_in_table` the row of that
detector's table, so the two together point at one hit. To read a field of
every hit of an event, `read_hit_field_by_tcm` follows those pointers and
returns the values with the same jagged structure as the TCM.

In [ ]:
energies = io.read_hit_field_by_tcm(tcm, hit_file, "energy")

# keep the events in which at least one germanium detector fired
event = tcm[ak.any(tcm.table_key != 3, axis=-1)]
energies = energies[ak.any(tcm.table_key != 3, axis=-1)]

# total energy in germanium and in argon, per event
germanium = ak.sum(energies[event.table_key != 3], axis=-1)
argon = ak.sum(energies[event.table_key == 3], axis=-1)

# how many germanium detectors fired in each event
multiplicity = ak.num(energies[event.table_key != 3], axis=-1)
multiplicity.show(limit_rows=1)

Most events fire a single germanium detector, and one in eight fires both:
those are the events in which a gamma scattered from one detector into the
other.

The argon says something about the rest of the decay. Rejecting the events that
deposited energy in it leaves only those in which everything that the decay
released stayed inside a germanium detector.

In [ ]:
veto = argon < 100  # unit is keV

fig, ax = plt.subplots(figsize=(12, 4))
for values, label in ((germanium, "all events"), (germanium[veto], "no energy in argon")):
    h = hist.new.Reg(160, 1000, 3400, name="energy [keV]").Double().fill(values)
    h.plot(ax=ax, yerr=False, flow="none", label=label)
ax.set_yscale("log")
ax.set_xlim(1000, 3400)
ax.set_ylabel("counts / 20 keV")
_ = ax.legend()

Only 16% of the events survive. The source sits in the middle of the argon, so
almost every decay leaves energy there. The 2615 keV line of $^{208}$Tl is cut
down as hard as the continuum, to 9%, because that gamma is emitted in cascade
with a 583 keV one: even when the 2615 keV gamma is fully absorbed in
germanium, its partner is usually stopped by the argon. The small bump at
3198 keV is the case where it is not: both gammas of the cascade absorbed in
germanium, which is what leaves the argon empty. A real experiment uses the
veto the other way around, to reject the background events that a source
deliberately produces here.

## Scaling up

Everything above held the whole file in memory. A production simulation is
larger than the memory of the machine, and then the file has to be read in
pieces. `LH5Iterator` returns one chunk of rows at a time, and
`write_hit_table_chunk` appends each result to the output file, so the loop
body is the same code as before.

In [ ]:
chunked_file = "th228_hit_chunked.lh5"
Path(chunked_file).unlink(missing_ok=True)

for stp in lh5.LH5Iterator(stp_file, "stp/ICPC", buffer_len=5000):
    data = stp.view_as("ak", with_units=True)
    out = io.init_hit_table(stp)

    distance = surface.distance_to_surface(
        data.xloc, data.yloc, data.zloc, hpges["ICPC"], positions["ICPC"]
    )
    activeness = functions.piecewise_linear_activeness(distance, fccd_in_mm=1.5, dlf=0.2)
    out.add_field(
        "energy",
        Array(ak.sum(data.edep * activeness, axis=-1), attrs={"units": "keV"}),
    )

    io.write_hit_table_chunk(out, "hit/ICPC", chunked_file, uid=2)

Two things to keep in mind. `write_hit_table_chunk` appends to whatever it
finds, so the output file has to be deleted before the first chunk, as above.
And the TCM has to be built after the loop, over the finished file, because a
coincidence can span two chunks.

A last note for the case where the work is split across several jobs. Chunks of
rows do not respect event boundaries: the hits of one event can fall in two
different chunks, and a job that reads a fixed number of rows can cut an event
in half. `reboost.io.get_rows_in_event_range` gives the first row and the
number of rows of a detector table that belong to a range of events, ready to
be passed to `LH5Iterator` as `i_start` and `n_entries`, so that each job reads
whole events. It needs the detector tables to be stored in event order, which
is the case for a simulation run on a single thread.